# 03 — Train Orthoptera Classifier
Fine-tune a CNN on the prepared label splits using OpenSoundscape, then export the trained model to `models/` for use in BASE.

**Kernel:** `Python (orthoptera-training)`  
**Prerequisites:** Run `02_prepare_labels.ipynb` first to generate `training/data/train.csv` etc.

In [15]:
import torch
import pandas as pd
from pathlib import Path
import opensoundscape.ml.cnn
from opensoundscape import CNN

PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR   = PROJECT_ROOT / "training" / "data"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

train = pd.read_csv(DATA_DIR / "train.csv", index_col=[0, 1, 2])
val   = pd.read_csv(DATA_DIR / "val.csv",   index_col=[0, 1, 2])
test  = pd.read_csv(DATA_DIR / "test.csv",  index_col=[0, 1, 2])

print(f"Classes: {list(train.columns)}")
print(f"Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")

Classes: ['Chorthippus brunneus brunneus', 'Pseudochorthippus parallelus parallelus', 'Omocestus viridulus', 'Tettigonia viridissima', 'Roeseliana roeselii', 'Pholidoptera griseoaptera', 'Leptophyes punctatissima', 'Gryllus campestris']
Train: 11629  Val: 3904  Test: 3137


In [2]:
# ── Build model ──────────────────────────────────────────────────────────────
model = CNN(
    architecture="resnet18",
    classes=list(train.columns),
    sample_duration=4.0,
)
print("Model built — architecture: resnet18")
print(f"Classes: {model.classes}")
model.network.to("mps")
print(next(model.network.parameters()).device)


Model built — architecture: resnet18
Classes: ['Chorthippus brunneus brunneus', 'Pseudochorthippus parallelus parallelus', 'Omocestus viridulus', 'Tettigonia viridissima', 'Roeseliana roeselii', 'Pholidoptera griseoaptera', 'Leptophyes punctatissima', 'Gryllus campestris']
mps:0


In [3]:
# ── Train ────────────────────────────────────────────────────────────────────
# Run config: 5 epochs, batch_size=32, num_workers=2, CPU.
# ~7-8h per epoch on CPU (~40h total). Use a GPU machine to cut this to minutes.

model.train(
    train,
    val,
    epochs=5,
    batch_size=32,
    save_path=MODELS_DIR / "orthoptera_checkpoints",
    save_interval=1,
    num_workers=2,
)



Training Epoch 0


  0%|          | 0/364 [00:00<?, ?it/s]

/Users/dawsonmccall/Documents/coding_projects/BioAcousticStreamEngine/.venv-training/lib/python3.11/site-packages/torch/utils/data/dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch: 0 [batch 0/364, 0.00%] 
	Epoch Running Average Loss: 0.593
	Most Recent Batch Loss: 0.593


/Users/dawsonmccall/Documents/coding_projects/BioAcousticStreamEngine/.venv-training/lib/python3.11/site-packages/opensoundscape/audio.py:341: UserWarning: Audio object is shorter than requested duration: 3.8213333333333335 sec instead of 4.0 sec
  warnings.warn(error_msg)
/Users/dawsonmccall/Documents/coding_projects/BioAcousticStreamEngine/.venv-training/lib/python3.11/site-packages/opensoundscape/audio.py:341: UserWarning: Audio object is shorter than requested duration: 3.764125 sec instead of 4.0 sec
  warnings.warn(error_msg)
/Users/dawsonmccall/Documents/coding_projects/BioAcousticStreamEngine/.venv-training/lib/python3.11/site-packages/opensoundscape/audio.py:341: UserWarning: Audio object is shorter than requested duration: 3.6436380208333334 sec instead of 4.0 sec
  warnings.warn(error_msg)


...

/Users/dawsonmccall/.pyenv/versions/3.11.9/lib/python3.11/tempfile.py:934: ResourceWarning: Implicitly cleaning up <TemporaryDirectory '/var/folders/dc/5p9wmvyd0m3c1sylx0z9lwlw0000gn/T/tmptse7codbwandb'>
  _warnings.warn(warn_message, ResourceWarning)
/Users/dawsonmccall/.pyenv/versions/3.11.9/lib/python3.11/tempfile.py:934: ResourceWarning: Implicitly cleaning up <TemporaryDirectory '/var/folders/dc/5p9wmvyd0m3c1sylx0z9lwlw0000gn/T/tmputunst62wandb'>
  _warnings.warn(warn_message, ResourceWarning)
/Users/dawsonmccall/.pyenv/versions/3.11.9/lib/python3.11/tempfile.py:934: ResourceWarning: Implicitly cleaning up <TemporaryDirectory '/var/folders/dc/5p9wmvyd0m3c1sylx0z9lwlw0000gn/T/tmp8qdawk3ewandb-artifacts'>
  _warnings.warn(warn_message, ResourceWarning)
/Users/dawsonmccall/.pyenv/versions/3.11.9/lib/python3.11/tempfile.py:934: ResourceWarning: Implicitly cleaning up <TemporaryDirectory '/var/folders/dc/5p9wmvyd0m3c1sylx0z9lwlw0000gn/T/tmp0phh2aumwandb-artifacts'>
  _warnings.warn(war

Metrics:
	MAP: 0.836

Validation.


  0%|          | 0/122 [00:00<?, ?it/s]

/Users/dawsonmccall/Documents/coding_projects/BioAcousticStreamEngine/.venv-training/lib/python3.11/site-packages/opensoundscape/audio.py:341: UserWarning: Audio object is shorter than requested duration: 3.917234375 sec instead of 4 sec
  warnings.warn(error_msg)
/Users/dawsonmccall/Documents/coding_projects/BioAcousticStreamEngine/.venv-training/lib/python3.11/site-packages/opensoundscape/audio.py:341: UserWarning: Audio object is shorter than requested duration: 3.6872192 sec instead of 4 sec
  warnings.warn(error_msg)
/Users/dawsonmccall/Documents/coding_projects/BioAcousticStreamEngine/.venv-training/lib/python3.11/site-packages/opensoundscape/audio.py:341: UserWarning: Audio object is shorter than requested duration: 3.6855808 sec instead of 4 sec
  warnings.warn(error_msg)
/Users/dawsonmccall/Documents/coding_projects/BioAcousticStreamEngine/.venv-training/lib/python3.11/site-packages/opensoundscape/audio.py:341: UserWarning: Audio object is shorter than requested duration: 3.68

Metrics:
	MAP: 0.563

Best Model Appears at Epoch 3 with Validation score 0.576.


In [6]:
# ── Evaluate on held-out test set (6,459 clips) ──────────────────────────────
from sklearn.metrics import classification_report
import numpy as np
from opensoundscape.ml.cnn import load_model

scores = model.predict(test)
y_true = test.values
y_pred = (scores.values > 0.5).astype(int)
print(classification_report(
    y_true, y_pred,
    target_names=list(train.columns),
    zero_division=0,
))


  0%|          | 0/3137 [00:00<?, ?it/s]

/Users/dawsonmccall/Documents/coding_projects/BioAcousticStreamEngine/.venv-training/lib/python3.11/site-packages/opensoundscape/audio.py:341: UserWarning: Audio object is shorter than requested duration: 3.6644791666666667 sec instead of 4 sec
  warnings.warn(error_msg)
/Users/dawsonmccall/Documents/coding_projects/BioAcousticStreamEngine/.venv-training/lib/python3.11/site-packages/opensoundscape/audio.py:341: UserWarning: Audio object is shorter than requested duration: 3.8499092970521542 sec instead of 4 sec
  warnings.warn(error_msg)
/Users/dawsonmccall/Documents/coding_projects/BioAcousticStreamEngine/.venv-training/lib/python3.11/site-packages/opensoundscape/audio.py:341: UserWarning: Audio object is shorter than requested duration: 3.9105729166666667 sec instead of 4 sec
  warnings.warn(error_msg)
/Users/dawsonmccall/Documents/coding_projects/BioAcousticStreamEngine/.venv-training/lib/python3.11/site-packages/opensoundscape/audio.py:341: UserWarning: Audio object is shorter than

                                         precision    recall  f1-score   support

          Chorthippus brunneus brunneus       0.93      0.34      0.50        41
Pseudochorthippus parallelus parallelus       0.89      0.37      0.53       295
                    Omocestus viridulus       0.97      0.55      0.70        64
                 Tettigonia viridissima       0.83      0.62      0.71       887
                    Roeseliana roeselii       0.51      0.50      0.50       693
              Pholidoptera griseoaptera       0.99      0.41      0.58       509
               Leptophyes punctatissima       0.61      0.15      0.24       238
                     Gryllus campestris       0.98      0.91      0.94       861

                              micro avg       0.80      0.58      0.67      3588
                              macro avg       0.84      0.48      0.59      3588
                           weighted avg       0.82      0.58      0.66      3588
                          

In [22]:
best_model = torch.load(MODELS_DIR / "orthoptera_checkpoints" / "best.model", weights_only=False)

scores = best_model.predict(test)
y_true = test.values
y_pred = (scores.values > 0.5).astype(int)
print(classification_report(
    y_true, y_pred,
    target_names=list(train.columns),
    zero_division=0,
))


  0%|          | 0/3137 [00:00<?, ?it/s]

/Users/dawsonmccall/Documents/coding_projects/BioAcousticStreamEngine/.venv-training/lib/python3.11/site-packages/opensoundscape/audio.py:341: UserWarning: Audio object is shorter than requested duration: 3.6644791666666667 sec instead of 4 sec
  warnings.warn(error_msg)
/Users/dawsonmccall/Documents/coding_projects/BioAcousticStreamEngine/.venv-training/lib/python3.11/site-packages/opensoundscape/audio.py:341: UserWarning: Audio object is shorter than requested duration: 3.8499092970521542 sec instead of 4 sec
  warnings.warn(error_msg)
/Users/dawsonmccall/Documents/coding_projects/BioAcousticStreamEngine/.venv-training/lib/python3.11/site-packages/opensoundscape/audio.py:341: UserWarning: Audio object is shorter than requested duration: 3.9105729166666667 sec instead of 4 sec
  warnings.warn(error_msg)
/Users/dawsonmccall/Documents/coding_projects/BioAcousticStreamEngine/.venv-training/lib/python3.11/site-packages/opensoundscape/audio.py:341: UserWarning: Audio object is shorter than

                                         precision    recall  f1-score   support

          Chorthippus brunneus brunneus       1.00      0.32      0.48        41
Pseudochorthippus parallelus parallelus       0.66      0.40      0.50       295
                    Omocestus viridulus       0.98      0.66      0.79        64
                 Tettigonia viridissima       0.83      0.54      0.65       887
                    Roeseliana roeselii       0.66      0.26      0.37       693
              Pholidoptera griseoaptera       0.90      0.41      0.56       509
               Leptophyes punctatissima       0.86      0.03      0.05       238
                     Gryllus campestris       0.96      0.94      0.95       861

                              micro avg       0.86      0.52      0.64      3588
                              macro avg       0.86      0.44      0.54      3588
                           weighted avg       0.83      0.52      0.61      3588
                          

In [ ]:
# ── Threshold sweep — find F1-maximising cutoff per species ──────────────────
import numpy as np
import pandas as pd

thresholds = np.arange(0.1, 0.91, 0.05)
results = []
for t in thresholds:
    y_pred_t = (scores.values > t).astype(int)
    from sklearn.metrics import f1_score
    macro_f1 = f1_score(y_true, y_pred_t, average='macro', zero_division=0)
    micro_f1 = f1_score(y_true, y_pred_t, average='micro', zero_division=0)
    results.append({'threshold': round(t, 2), 'macro_f1': round(macro_f1, 3), 'micro_f1': round(micro_f1, 3)})

sweep_df = pd.DataFrame(results)
best = sweep_df.loc[sweep_df['macro_f1'].idxmax()]
print(f"Best macro-F1 threshold: {best['threshold']}  (macro={best['macro_f1']}, micro={best['micro_f1']})")
print()
print(sweep_df.to_string(index=False))


### Results — 5-epoch ResNet18 (CPU, May 2026)

| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima* | 0.96 | 0.61 | **0.75** | 2,530 |
| *Chorthippus brunneus brunneus* | 0.81 | 0.23 | **0.36** | 73 |
| *Pholidoptera griseoaptera* | 0.76 | 0.12 | **0.21** | 1,393 |
| *Leptophyes punctatissima* | 0.85 | 0.08 | **0.15** | 359 |
| *Gryllus campestris* | 0.94 | 0.06 | **0.12** | 929 |
| *Roeseliana roeselii* | 0.59 | 0.06 | **0.10** | 741 |
| *Pseudochorthippus parallelus* | 0.77 | 0.03 | **0.05** | 358 |
| *Omocestus viridulus* | 1.00 | 0.01 | **0.03** | 76 |

**Macro F1: 0.22 — precision is high (low false-positive rate) but recall is low.**

Pattern: the model is conservative after only 5 epochs — it fires confidently when it fires,
but misses most detections. *Tettigonia* performs well (most training data: 6,270 clips).

**To improve:**
- More epochs (20–30) — the main lever
- Lower `min_confidence` threshold in `config/settings.yaml` (try 0.3)
- GPU training would make 30 epochs feasible in <1 hour

**Threshold sweep verdict:** macro-F1 is flat across all thresholds (0.215–0.225). Lowering the threshold does not help — the model's score distribution is too compressed after only 5 epochs. More training epochs is the only meaningful lever.


### Results — 5-epoch ResNet18 (CPU, June 23rd 2026)
### Results after fixes for Multi-hot Encoding and proper start and end times

| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima* | 0.83 | 0.62 | **0.71** | 887 |
| *Chorthippus brunneus brunneus* | 0.93 | 0.34 | **0.50** | 41 |
| *Pholidoptera griseoaptera* | 0.99 | 0.41 | **0.58** | 509 |
| *Leptophyes punctatissima* | 0.61 | 0.15 | **0.24** | 238 |
| *Gryllus campestris* | 0.98 | 0.91 | **0.94** | 861 |
| *Roeseliana roeselii* | 0.51 | 0.50 | **0.50** | 693 |
| *Pseudochorthippus parallelus* | 0.89 | 0.37 | **0.53** | 295 |
| *Omocestus viridulus* | 0.97 | 0.55 | **0.70** | 64 |

**Macro F1: 0.59.**


In [ ]:
# ── Save final model ──────────────────────────────────────────────────────────
# OpenSoundscape saves as a .model file (a torch pickle).
# This is the file you point insect.py at in BASE.

model_path = MODELS_DIR / "orthoptera_uk.model"
model.save(model_path)
print(f"Model saved: {model_path}")
print()
print("Next step: set 'model_path' in config/settings.yaml under 'insect:'")
print("and activate 'insect' in classifiers.active")